<a href="https://colab.research.google.com/github/rhiosutoyo/Teaching-Deep-Learning-and-Its-Applications/blob/main/06_2_exploring_transfer_learning_with_pretrained_resnet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exploring Transfer Learning with Pretrained ResNet


In [1]:
# 1. Import Necessary Libraries
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50V2
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt

 This chapter's goal is to load the CIFAR-10 data and transform it into a standardized format that the ResNet model can understand. This involves resizing, augmenting, and organizing the data into efficient batches.



In [2]:
# 2. Load and Prepare the Dataset
print("Loading CIFAR-10 dataset...")
(train_ds, val_ds, test_ds), ds_info = tfds.load(
    'cifar10',
    split=['train', 'test[:50%]', 'test[50%:]'], # Use test set for validation and testing
    shuffle_files=True,
    as_supervised=True, # Returns (image, label) tuples
    with_info=True,
)

# --- Preprocessing ---
IMG_SIZE = 224 # ResNet models expect a larger input size than CIFAR-10's 32x32
BATCH_SIZE = 32

def preprocess(image, label):
    """Resizes and normalizes images for the model."""
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.keras.applications.resnet_v2.preprocess_input(image)
    return image, label

# Apply preprocessing to the datasets and create batches
train_ds = train_ds.map(preprocess).cache().shuffle(ds_info.splits['train'].num_examples).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.map(preprocess).batch(BATCH_SIZE).cache().prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.map(preprocess).batch(BATCH_SIZE).cache().prefetch(tf.data.AUTOTUNE)

Loading CIFAR-10 dataset...


We load an expert model, freeze its existing knowledge, and replace its final layer with a new, untrained one that is specific to our task.

In [3]:
# 3. Build the Model
print("Building the model with ResNet50V2 base...")

# Load the ResNet50V2 model, pre-trained on ImageNet
# include_top=False means we don't include the final classification layer
base_model = ResNet50V2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False, # This command instructs Keras to leave off the original final layer of the ResNet model. That layer was trained to classify 1,000 specific ImageNet categories, which aren't relevant for our 10 CIFAR-10 classes.
    weights='imagenet' # This command tells Keras to load the ResNet model with all the "knowledge" (weights) it has already gained from being trained on the massive ImageNet dataset.
)

# Freeze the base model layers so we don't train them
base_model.trainable = False # This prevents the model from altering or destroying the valuable, pre-existing knowledge from ImageNet during our new training process.

# --- Create the new model head ---
# We add our own new layers on top of the frozen base
inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False) # Run the base model in inference mode
x = layers.GlobalAveragePooling2D()(x) # Pool the features
x = layers.Dropout(0.2)(x)             # Add a dropout layer for regularization
outputs = layers.Dense(10)(x)          # Add the final classification layer for 10 classes | This is the only part of the model that starts with no knowledge.

# Combine the base model and new layers into one final model
model = keras.Model(inputs, outputs)

print("Model summary:")
model.summary()

Building the model with ResNet50V2 base...
Model summary:


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50v2 (Functional)         │ (None, 7, 7, 2048)     │    23,564,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │        20,490 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,585,290 (89.97 MB)

 Trainable params: 20,490 (80.04 KB)

 Non-trainable params: 23,564,800 (89.89 MB)

Before training, we must define the tools the model will use to learn. This involves choosing a method to measure the model's error (loss) and an algorithm to update its weights (optimizer) to reduce that error.

In [4]:
# 4. Compile the Model
print("\nCompiling the model...")
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)


Compiling the model...


This section contains the main engine of the learning process. It's a function that loops through the data for a set number of cycles (epochs), feeding the data to the model, calculating its error, and telling the optimizer to update the model's weights.

In [5]:
# 5. Train the Model
print("\nStarting training... 🚀")
history = model.fit(
    train_ds,
    epochs=5, # Train for 5 epochs for a quick demo
    validation_data=val_ds
)


Starting training... 🚀
Epoch 1/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 63s 28ms/step - accuracy: 0.7751 - loss: 0.6586 - val_accuracy: 0.8682 - val_loss: 0.3781
Epoch 2/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 30s 19ms/step - accuracy: 0.8768 - loss: 0.3592 - val_accuracy: 0.8774 - val_loss: 0.3711
Epoch 3/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 29s 19ms/step - accuracy: 0.8878 - loss: 0.3385 - val_accuracy: 0.8722 - val_loss: 0.3764
Epoch 4/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 29s 19ms/step - accuracy: 0.8925 - loss: 0.3192 - val_accuracy: 0.8790 - val_loss: 0.3658
Epoch 5/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 29s 19ms/step - accuracy: 0.8950 - loss: 0.3078 - val_accuracy: 0.8810 - val_loss: 0.3803


In [6]:
# 6. Evaluate the Model
print("\nTraining finished! Evaluating on test data...")
loss, accuracy = model.evaluate(test_ds)
print(f"Test Accuracy: {accuracy:.2%}")


Training finished! Evaluating on test data...
157/157 ━━━━━━━━━━━━━━━━━━━━ 4s 27ms/step - accuracy: 0.8798 - loss: 0.3756
Test Accuracy: 87.56%
